In [ ]:
# excel 前兩行數據
# MallCode,	OR_Landmark_ID,	MallNameTc,	MallNameEn,	Id,	OR_POI_ID,	ReferenceId,	MallId,	Name_en,	Name_tc,	Name_sc,	CategoryNameEn,	ShopNumber
# tmtplaza,	42,	屯門市廣場,	tmtplaza,	1592,	803423,	540,	1,	&you,	&you,	&you,	Western Cuisine,	3212
# tmtplaza,	42,	屯門市廣場,	tmtplaza,	1605,	794645,	544,	1,	A-1 Bakery Premium,	A-1 Bakery Premium,	A-1 Bakery Premium,	Confectionery & Deli,	UG046

# 我的需求如下
# 1. 讀取Sino POI full list.xlsx的sheet Sino POIs，根據每個landmark id 分組，統計每個landmark id 下的相應poi的以下數據：
# No. of POIs - Booking	
# No. of Booking	
# No. of POIs - Voucher	
# No. of Voucher Transaction	
# No. of POIs - Takeaway	
# No. of Takeaway Transaction	
# Retail POIs Video Views
# 代碼請參考 Sino_poi.sql(請注意，代碼是基於landmark所有商戶統計的，我要的是在excel裏landmark含有的poi：指定商戶)
# 請修改代碼，讓他計算excel的poi的統計數據，而且我需要一次性呈現所有結果（像sql一樣），df只能有一個結果，不方便。
# 請保持SET @landmarkId = 37 的設定，我可以自己輸入landmark id生成結果。


In [ ]:
import os

import pandas as pd
import pymssql

DB_info = {
    'server': '192.168.61.119:7622',
    'user': 'BAReporting',
    'password': 'KeHeCReme8he',
}

START_DATE = '2026-09-01'
END_DATE = '2026-10-01'
LANDMARK_ID = 37  # Set to None to report every landmark in the Excel file.

excel_path = 'Sino POI full list.xlsx'
poi_df = pd.read_excel(excel_path, sheet_name='Sino POIs')

required_columns = ['OR Landmark ID', 'OR POI ID']
missing_columns = set(required_columns).difference(poi_df.columns)
if missing_columns:
    raise ValueError(f'Excel is missing required columns: {sorted(missing_columns)}')

selected_pois = (
    poi_df.rename(columns={
        'OR Landmark ID': 'LandmarkId',
        'OR POI ID': 'ORPoiId',
        'MallNameTc': 'MallNameTc',
        'MallNameEn': 'MallNameEn',
    })[['LandmarkId', 'ORPoiId', 'MallNameTc', 'MallNameEn']]
    .assign(
        LandmarkId=lambda frame: pd.to_numeric(frame['LandmarkId'], errors='coerce'),
        ORPoiId=lambda frame: pd.to_numeric(frame['ORPoiId'], errors='coerce'),
    )
    .dropna(subset=['LandmarkId', 'ORPoiId'])
    .astype({'LandmarkId': 'int64', 'ORPoiId': 'int64'})
    .drop_duplicates(subset=['LandmarkId', 'ORPoiId'])
)

if LANDMARK_ID is not None:
    selected_pois = selected_pois[selected_pois['LandmarkId'].eq(LANDMARK_ID)]

if selected_pois.empty:
    raise ValueError('No valid POIs found for the selected LANDMARK_ID.')

selected_pois

In [ ]:
with pymssql.connect(DB_info['server'], DB_info['user'], DB_info['password']) as conn:
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE #SelectedPoi (
            LandmarkId INT NOT NULL,
            ORPoiId INT NOT NULL,
            PRIMARY KEY (LandmarkId, ORPoiId)
        );
    ''')
    cursor.executemany(
        'INSERT INTO #SelectedPoi (LandmarkId, ORPoiId) VALUES (%s, %s)',
        list(selected_pois[['LandmarkId', 'ORPoiId']].itertuples(index=False, name=None)),
    )

    sql = '''
    DECLARE @START DATE = %s;
    DECLARE @END DATE = %s;

    ;WITH LandmarkList AS (
        SELECT
            LandmarkId,
            MAX(MallNameTc) AS MallNameTc,
            MAX(MallNameEn) AS MallNameEn,
            COUNT(*) AS Excel_POI_Count
        FROM (
            SELECT DISTINCT
                LandmarkId,
                ORPoiId,
                MallNameTc,
                MallNameEn
            FROM #SelectedPoi selected_poi
            INNER JOIN (
                SELECT DISTINCT
                    LandmarkId,
                    ORPoiId,
                    MallNameTc,
                    MallNameEn
                FROM #SelectedPoi selected_ids
                INNER JOIN mars.dbo.Poi mars_poi
                    ON mars_poi.ORPoiId = selected_ids.ORPoiId
                   AND mars_poi.RegionId = 0
            ) mapped_names
                ON mapped_names.LandmarkId = selected_poi.LandmarkId
               AND mapped_names.ORPoiId = selected_poi.ORPoiId
        ) named_pois
        GROUP BY LandmarkId
    ),
    MappedPoi AS (
        SELECT DISTINCT
            selected_poi.LandmarkId,
            mars_poi.PoiId AS MarsPoiId,
            selected_poi.ORPoiId
        FROM #SelectedPoi selected_poi
        INNER JOIN mars.dbo.Poi mars_poi
            ON mars_poi.ORPoiId = selected_poi.ORPoiId
           AND mars_poi.RegionId = 0
    ),
    BookingStats AS (
        SELECT
            mapped_poi.LandmarkId,
            COUNT(DISTINCT booking.PoiId) AS Booking_POI_Count,
            COUNT(*) AS Booking_Count
        FROM MappedPoi mapped_poi
        INNER JOIN mars.dbo.Booking booking
            ON booking.PoiId = mapped_poi.MarsPoiId
           AND booking.BookingTime >= @START
           AND booking.BookingTime < @END
           AND booking.Status = 10
        GROUP BY mapped_poi.LandmarkId
    ),
    VoucherServiceStats AS (
        SELECT
            mapped_poi.LandmarkId,
            COUNT(DISTINCT mapped_poi.MarsPoiId) AS Voucher_POI_Count
        FROM MappedPoi mapped_poi
        INNER JOIN mars.dbo.BizService service
            ON service.PoiId = mapped_poi.MarsPoiId
           AND service.ServiceTypeId = 5
           AND service.ServiceStartTime < @END
           AND service.ServiceEndTime >= @END
        GROUP BY mapped_poi.LandmarkId
    ),
    VoucherStats AS (
        SELECT
            mapped_poi.LandmarkId,
            COUNT(*) AS Voucher_Transaction_Count
        FROM MappedPoi mapped_poi
        INNER JOIN mars.dbo.OfferWallet wallet
            ON wallet.RedeemPoiId = mapped_poi.MarsPoiId
           AND wallet.CreateTime >= @START
           AND wallet.CreateTime < @END
           AND wallet.Status = 15
        INNER JOIN mars.dbo.Offer offer
            ON offer.OfferId = wallet.OfferId
           AND offer.OfferType IN (7, 18, 22)
        WHERE EXISTS (
            SELECT 1
            FROM mars.dbo.OfferPoi offer_poi
            WHERE offer_poi.OfferId = wallet.OfferId
              AND offer_poi.PoiId = mapped_poi.MarsPoiId
        )
        GROUP BY mapped_poi.LandmarkId
    ),
    TakeawayStats AS (
        SELECT
            mapped_poi.LandmarkId,
            COUNT(DISTINCT takeaway.PoiId) AS Takeaway_POI_Count,
            COUNT(*) AS Takeaway_Transaction_Count
        FROM MappedPoi mapped_poi
        INNER JOIN mars.dbo.TakeAwayOrder takeaway WITH (NOLOCK)
            ON takeaway.PoiId = mapped_poi.MarsPoiId
           AND takeaway.CreateTime >= @START
           AND takeaway.CreateTime < @END
           AND takeaway.Status = 10
        GROUP BY mapped_poi.LandmarkId
    ),
    RetailVideoStats AS (
        SELECT
            selected_poi.LandmarkId,
            SUM(COALESCE(media.HitCount, 0)) AS Retail_Video_Views
        FROM #SelectedPoi selected_poi
        INNER JOIN openrice3.dbo.Media media
            ON media.PoiId = selected_poi.ORPoiId
        INNER JOIN openrice3.dbo.Poi poi
            ON poi.PoiId = media.PoiId
           AND poi.PoiTypeId = 10
        GROUP BY selected_poi.LandmarkId
    )
    SELECT
        landmarks.LandmarkId,
        landmarks.MallNameTc,
        landmarks.MallNameEn,
        landmarks.Excel_POI_Count,
        COALESCE(booking.Booking_POI_Count, 0) AS No_of_POIs_Booking,
        COALESCE(booking.Booking_Count, 0) AS No_of_Bookings,
        COALESCE(voucher_service.Voucher_POI_Count, 0) AS No_of_POIs_Voucher,
        COALESCE(voucher.Voucher_Transaction_Count, 0) AS No_of_Voucher_Transactions,
        COALESCE(takeaway.Takeaway_POI_Count, 0) AS No_of_POIs_Takeaway,
        COALESCE(takeaway.Takeaway_Transaction_Count, 0) AS No_of_Takeaway_Transactions,
        COALESCE(retail_video.Retail_Video_Views, 0) AS Retail_POIs_Video_Views
    FROM LandmarkList landmarks
    LEFT JOIN BookingStats booking ON booking.LandmarkId = landmarks.LandmarkId
    LEFT JOIN VoucherServiceStats voucher_service ON voucher_service.LandmarkId = landmarks.LandmarkId
    LEFT JOIN VoucherStats voucher ON voucher.LandmarkId = landmarks.LandmarkId
    LEFT JOIN TakeawayStats takeaway ON takeaway.LandmarkId = landmarks.LandmarkId
    LEFT JOIN RetailVideoStats retail_video ON retail_video.LandmarkId = landmarks.LandmarkId
    ORDER BY landmarks.LandmarkId;
    '''

    report_df = pd.read_sql(sql, conn, params=(START_DATE, END_DATE))

report_df